# M³TrustEval-XQuAD  
## Modular Multilingual Trustworthiness Evaluation of a HuggingFace LLM

### Course Project: Large Language Models

---

## 1. Project Summary

In this project, you will evaluate a single HuggingFace instruction-tuned language model on a real multilingual question-answering dataset.

The project focuses on five advanced LLM topics:

1. **Evaluation**
2. **Interpretability**
3. **Bias / Ethics**
4. **Modularity**
5. **Multilinguality**

The key idea is that LLM evaluation should not be limited to exact answer accuracy.  
A trustworthy evaluation should also ask:

- Does the model answer correctly?
- Is the explanation grounded in the given context?
- Does a modular verifier improve the answer?
- Does the model perform equally well across languages?
- Are there language-based performance disparities that raise fairness and ethical concerns?

---

## 2. Dataset and Model

### Dataset

We use **XQuAD** from HuggingFace:

- Dataset: `google/xquad`
- Task: Cross-lingual extractive question answering
- Languages used in this notebook: English, Arabic, German, Spanish, Turkish

XQuAD is useful here because the same QA examples are translated across multiple languages.  
This allows us to compare the same model across languages in a controlled way.

### Model

We use exactly one HuggingFace model:

- Model: `Qwen/Qwen2.5-0.5B-Instruct`

This is intentionally a small model so that the notebook can run on limited GPU resources.


# 3. Reference Papers and Links

This project is inspired by the following papers and resources.

## Main Evaluation Reference

**HELM: Holistic Evaluation of Language Models**  
Link: https://arxiv.org/abs/2211.09110

HELM motivates the idea that language models should be evaluated across multiple metrics, not just accuracy.

## Trustworthiness Reference

**DecodingTrust: A Comprehensive Assessment of Trustworthiness in GPT Models**  
Link: https://arxiv.org/abs/2306.11698

DecodingTrust motivates the trustworthiness perspective, including fairness, robustness, ethics, privacy, and bias.

## Modularity / Self-Consistency Reference

**SelfCheckGPT: Zero-Resource Black-Box Hallucination Detection for Generative LLMs**  
Link: https://arxiv.org/abs/2303.08896

SelfCheckGPT motivates checking model outputs through consistency and verification.

## Dataset Reference

**XQuAD: Cross-lingual Question Answering Dataset**  
HuggingFace: https://huggingface.co/datasets/google/xquad  
GitHub: https://github.com/google-deepmind/xquad

## Model Reference

**Qwen2.5-0.5B-Instruct**  
HuggingFace: https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct


# 4. Formal Project Statement

## Goal

Build a multilingual and modular evaluation framework for a HuggingFace LLM using the XQuAD dataset.

## Required Tasks

Students must:

1. Load the XQuAD dataset from HuggingFace.
2. Select a small multilingual subset from multiple languages.
3. Load one HuggingFace instruction-tuned model.
4. Run a **direct answering baseline**.
5. Run a **modular pipeline** consisting of:
   - generator
   - verifier
   - reviser
6. Compute answer quality metrics:
   - Exact Match
   - Token-level F1
7. Compute interpretability / faithfulness metrics:
   - whether the evidence appears in the context
   - whether the explanation is grounded in the context
8. Compute multilingual fairness metrics:
   - per-language performance
   - English vs. non-English gap
   - worst-language performance
   - language performance disparity
9. Compare the direct and modular systems.
10. Write a short discussion about bias, ethics, limitations, and possible improvements.

## Final Deliverables

Students should submit:

- completed notebook,
- generated CSV files,
- plots,
- final written analysis.

## Important Rule

Use only one LLM for generation and verification:

`Qwen/Qwen2.5-0.5B-Instruct`


# 5. Expected Research Questions

Your final report should answer:

1. Does the modular pipeline improve performance over the direct baseline?
2. Does the verifier improve explanation faithfulness?
3. Which language has the lowest performance?
4. Is there a large performance gap between English and non-English languages?
5. Can cross-lingual performance disparity be considered a form of language bias?
6. What are the ethical implications of deploying a model that performs much better in some languages than others?
7. What are the limitations of using the same model as both generator and verifier?
8. How could this project be extended to become a stronger research study?


# 6. Install Required Libraries

Run this cell in Colab, Kaggle, or a fresh local environment.

Recommended runtime:

- GPU enabled
- Internet enabled


In [ ]:
# Run this cell to install the required libraries.
!pip -q install datasets transformers accelerate torch pandas numpy matplotlib tqdm rapidfuzz tabulate


# 7. Imports and Configuration

This notebook intentionally uses only one HuggingFace LLM.

You may reduce `N_EXAMPLES_PER_LANGUAGE` for quick debugging.  
For the final run, use a larger number if your GPU allows it.


In [ ]:
# TODO: import the required libraries and define the configuration variables.
# Required variables:
# - MODEL_ID
# - DATASET_ID
# - LANG_CONFIGS
# - N_EXAMPLES_PER_LANGUAGE
# - MAX_CONTEXT_CHARS
# - MAX_NEW_TOKENS


# 8. Load the XQuAD Dataset

We load the same dataset in several languages.  
Then we keep examples that have matching IDs across languages, so that the same underlying QA item can be compared cross-lingually.


In [ ]:
# TODO:
# 1. Load XQuAD for each language in LANG_CONFIGS.
# 2. Extract the first gold answer from the answers field.
# 3. Keep examples with common IDs across languages.
# 4. Return a DataFrame with:
#    id, lang, title, context, question, gold_answer


# 9. Load the HuggingFace Model

This notebook uses only one model:

`Qwen/Qwen2.5-0.5B-Instruct`

The same model is used for:

- direct answering,
- answer generation,
- verification,
- revision.

This makes the project simple and reproducible, but it also creates an important limitation: the verifier may share the same weaknesses as the generator.


In [ ]:
# TODO:
# Load the tokenizer and model from MODEL_ID using HuggingFace Transformers.
# Recommended:
# - AutoTokenizer.from_pretrained
# - AutoModelForCausalLM.from_pretrained
# - device_map="auto"


# 10. Helper Functions for Text Normalization and JSON Parsing

The model is asked to return JSON, but small LLMs may not always produce perfect JSON.  
The functions below make the evaluation more robust.


In [ ]:
# TODO:
# Implement helper functions:
# - extract_json
# - normalize_answer
# - token_f1
# - exact_match
# - fuzzy_match


# 11. Model Generation Function

This function sends a prompt to the model and returns the generated text.

The function uses the model's chat template when available.


In [ ]:
# TODO:
# Implement a function that:
# 1. receives a prompt,
# 2. formats it as a chat message,
# 3. tokenizes it,
# 4. calls model.generate,
# 5. returns the generated text.


# 12. Direct Answering Prompt

The direct baseline asks the model to answer using only the context.

The output must contain:

- `answer`
- `explanation`
- `evidence`
- `confidence`


In [ ]:
# TODO:
# Implement:
# - build_direct_prompt
# - run_direct
#
# The model should return JSON with:
# answer, explanation, evidence, confidence


# 13. Run the Direct Baseline

This cell may take time depending on the number of examples and your hardware.


In [ ]:
# TODO:
# Run the direct baseline on all examples and store the results in direct_df.


# 14. Modular Pipeline Prompt

The modular pipeline has three stages:

1. **Generator**: produces an initial answer.
2. **Verifier**: checks whether the answer is supported by the context.
3. **Reviser**: produces a corrected final answer if needed.

For simplicity, this notebook uses the same HuggingFace model for all stages.


In [ ]:
# TODO:
# Implement:
# - build_verifier_prompt
# - run_modular
#
# The modular system should:
# 1. generate an initial answer,
# 2. verify it,
# 3. revise it if needed,
# 4. return the final answer.


# 15. Run the Modular Pipeline


In [ ]:
# TODO:
# Run the modular pipeline on all examples and store the results in modular_df.


# 16. Compute Evaluation Metrics

We compute:

- Exact Match
- Token F1
- Fuzzy Match
- Evidence faithfulness
- Explanation faithfulness

The faithfulness metrics are simple approximations.  
They check whether the evidence and explanation are grounded in the context.


In [ ]:
# TODO:
# Implement scoring functions for:
# - exact_match
# - token_f1
# - fuzzy_match
# - evidence_in_context
# - answer_in_explanation
# - faithfulness
#
# Then apply them to direct_df and modular_df.


# 17. Aggregate Results by Mode and Language

This table is the main evaluation output of the project.


In [ ]:
# TODO:
# Group results by mode and language.
# Report average Exact Match, Token F1, Fuzzy Match, Faithfulness, and Evidence Rate.


# 18. Compare Direct vs. Modular Systems

This table shows whether the modular pipeline improved over the direct baseline.


In [ ]:
# TODO:
# Group results by mode and compare direct vs modular performance.


# 19. Multilingual Fairness and Language-Bias Metrics

In this project, **bias** is studied as language-based performance disparity.

If the model performs much better in English than in Arabic, Turkish, Spanish, or German, this may indicate a form of language bias or unequal service quality.

We compute:

- English vs. non-English performance gap
- best-language vs. worst-language gap
- standard deviation across languages
- worst-language performance


In [ ]:
# TODO:
# Implement language fairness metrics:
# - English vs non-English gap
# - best vs worst language gap
# - standard deviation across languages
# - worst language


# 20. Cross-Lingual Consistency

Because XQuAD contains parallel examples, we can compare whether the model is correct or incorrect on the same question across different languages.

A simple consistency metric:

> For each question ID, calculate how consistent the model's correctness is across languages.

If a model answers the English version correctly but fails the Arabic or Turkish version, this indicates cross-lingual inconsistency.


In [ ]:
# TODO:
# Compute cross-lingual consistency:
# For each question ID, check whether correctness is stable across languages.


# 21. Plot Results

Create simple plots for:

1. Token F1 by language and mode
2. Faithfulness by language and mode
3. Language disparity metrics
4. Cross-lingual consistency


In [ ]:
# TODO:
# Create plots for:
# 1. Token F1 by language and mode
# 2. Faithfulness by language and mode
# 3. Language disparity metrics
# 4. Cross-lingual consistency


# 22. Save Outputs

Save the main results as CSV files.


In [ ]:
# TODO:
# Save all important outputs as CSV files.


# 23. Generate Final Report

The final report should summarize:

- model and dataset,
- direct vs modular performance,
- per-language results,
- interpretability findings,
- language-bias / fairness findings,
- limitations,
- possible extensions.


In [ ]:
# TODO:
# Generate a final Markdown report summarizing the project results.


# 24. Final Written Analysis

Write your final analysis below.

Your answer should include:

1. A short description of the project.
2. A short explanation of HELM and why holistic evaluation matters.
3. A comparison of direct vs modular results.
4. A discussion of explanation faithfulness.
5. A discussion of multilingual performance.
6. A discussion of language bias and ethical implications.
7. At least three limitations.
8. At least three possible extensions.


In [ ]:
# TODO:
# Write your final analysis here.


# 25. Suggested Grading Rubric

| Component | Points |
|---|---:|
| Correct dataset loading and preprocessing | 10 |
| Correct HuggingFace model loading | 10 |
| Direct baseline implementation | 10 |
| Modular pipeline implementation | 20 |
| Evaluation metrics | 15 |
| Interpretability / faithfulness analysis | 10 |
| Multilingual fairness and bias/ethics analysis | 15 |
| Plots and saved outputs | 5 |
| Final written discussion | 5 |

Total: 100
